In [1]:
import sys
!{sys.executable} -m pip install lightgbm Levenshtein scikit-learn pandas numpy

# Imports (will execute immediately after installation in the same environment)
import os
import re
import numpy as np
import pandas as pd
import Levenshtein
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Imports successful! Everything is ready.")

Imports successful! Everything is ready.


In [2]:
# Create required output directory
os.makedirs("output", exist_ok=True)

# Load Training Data
train_s1 = pd.read_csv("dataset/train/train_source1.tsv", sep="\t")
train_s2 = pd.read_csv("dataset/train/train_source2.tsv", sep="\t")
train_s3 = pd.read_csv("dataset/train/train_source3.tsv", sep="\t")
train_gt = pd.read_csv("dataset/train/train_ground_truth.tsv", sep="\t")

# Load Test Data
test_s1 = pd.read_csv("dataset/test/test_source1.tsv", sep="\t")
test_s2 = pd.read_csv("dataset/test/test_source2.tsv", sep="\t")
test_s3 = pd.read_csv("dataset/test/test_source3.tsv", sep="\t")

print("Data loaded successfully!")

Data loaded successfully!


In [5]:
def clean_text_pure_python(text):
    if pd.isna(text):
        return ""
    text = str(text).lower().strip()
    text = re.sub(r'\b(corporation|corp)\b', 'corp', text)
    text = re.sub(r'\b(private|pvt)\b', 'pvt', text)
    text = re.sub(r'\b(limited|ltd)\b', 'ltd', text)
    text = re.sub(r'\s+and\s+', ' & ', text)
    text = re.sub(r'[^a-z0-9\s&]', '', text)
    return re.sub(r'\s+', ' ', text).strip()

def clean_series_safe(series, batch_size=50000):
    cleaned_list = []
    total_len = len(series)
    
# Process in safe small batches using pure Python loops
    for start_idx in range(0, total_len, batch_size):
        batch = series.iloc[start_idx:start_idx + batch_size]
        cleaned_batch = [clean_text_pure_python(x) for x in batch]
        cleaned_list.extend(cleaned_batch)
        
    return cleaned_list

# Apply safely across all dataframes
for df in [train_s1, train_s2, train_s3, test_s1, test_s2, test_s3]:
    print(f"Cleaning dataframe securely...")
    df['clean_name'] = clean_series_safe(df['business_name'])
    df['clean_address'] = clean_series_safe(df['business_address'])

print("Text preprocessing completed successfully without memory errors!")

Cleaning dataframe securely...
Cleaning dataframe securely...


MemoryError: Unable to allocate 4.80 MiB for an array with shape (5034616,) and data type uint8